###Create landing layer tables from parquet file path ###


In [0]:
# 1. Define the old and new paths
old_path = "/mnt/data_analytics_training/customers/Day-1"
new_path = "/mnt/data_analytics_training/customers_landing/Day-1"
df_customers_day1 = spark.read.parquet(old_path)
df_customers_day1.write.format("parquet").mode("overwrite").save(new_path)



In [0]:
# A list of tuples, where each tuple contains the DataFrame and its name


dataframes_to_process = [
     "orderdetails",
    "orders",
    "payments",
    "products"
]

# The base path for your training data
base_path = "/mnt/data_analytics_training"
# Loop through each DataFrame and write it to a new 'Day-2' directory
for  name in dataframes_to_process:
    # Construct the new path by replacing 'Day-1' with 'Day-2' or just creating it
    old_path = f"{base_path}/{name}/Day-1"
    new_path = f"{base_path}/{name}_landing/Day-1"
    df = spark.read.parquet(old_path)
    df.write.format("parquet").mode("overwrite").save(new_path)
    print(f"Writing {name} data to {new_path}...")
    print(f"Successfully wrote {name} data.")

print("\nAll DataFrames have been written to their new 'Day-1' locations.")


####Create Bronze from Landing#####

        ###1 Create schema####

In [0]:
%sql
DROP SCHEMA IF EXISTS bronze_138080 CASCADE;
create schema bronze_138080;
CREATE TABLE customers (
  customerNumber INT NOT NULL,
  customerName STRING NOT NULL,
  contactLastName STRING NOT NULL,
  contactFirstName STRING NOT NULL,
  phone STRING NOT NULL,
  addressLine1 STRING NOT NULL,
  addressLine2 STRING,
  city STRING NOT NULL,
  state STRING,
  postalCode STRING,
  country STRING NOT NULL,
  salesRepEmployeeNumber INT,
  creditLimit DECIMAL(10,2)
);
CREATE TABLE orderdetails (
  orderNumber INT NOT NULL,
  productCode VARCHAR(15) NOT NULL,
  quantityOrdered INT NOT NULL,
  priceEach DECIMAL(10,2) NOT NULL,
  orderLineNumber SMALLINT NOT NULL
);

CREATE TABLE orders (
  orderNumber INT NOT NULL,
  orderDate DATE NOT NULL,
  requiredDate DATE NOT NULL,
  shippedDate DATE,
  status VARCHAR(15) NOT NULL,
  comments STRING,
  customerNumber INT NOT NULL
);

CREATE TABLE payments (
  customerNumber INT NOT NULL,
  creditcardNumber VARCHAR(50) NOT NULL,
  paymentDate DATE NOT NULL,
  amount DECIMAL(10,2) NOT NULL
);

CREATE TABLE products (
  productCode VARCHAR(15) NOT NULL,
  productName VARCHAR(70) NOT NULL,
  productLine VARCHAR(50) NOT NULL,
  productScale VARCHAR(10) NOT NULL,
  productVendor VARCHAR(50) NOT NULL,
  productDescription STRING NOT NULL,
  quantityInStock SMALLINT NOT NULL,
  buyPrice DECIMAL(10,2) NOT NULL,
  MSRP DECIMAL(10,2) NOT NULL
);




      ###ingest Raw from parquet file##

In [0]:

dataframes_to_process = [
    "customers",
     "orderdetails",
    "orders",
    "payments",
    "products"
]
base_path = "/mnt/data_analytics_training"
# Loop through each DataFrame and write it to a new 'Day-2' directory
for  name in dataframes_to_process:
    parquet_path = f"{base_path}/{name}_landing/Day-1"
    table_name = f"bronze_138080.{name}"   
    df = spark.read.parquet(parquet_path)
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f"Successfully ingested data from {parquet_path} into table {table_name}")


### Silver Layer########

     #### customer##

      ### Customer

In [0]:
%sql
DROP SCHEMA IF EXISTS Silver_138080 CASCADE;
create schema Silver_138080;
CREATE TABLE Silver_138080.customers (
  -- SCD Type 2 Columns for History Tracking
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- Surrogate key for each version
  eff_start_date DATE NOT NULL,                      -- Date the record version became active
  eff_end_date DATE,                               -- Date the record version expired
  is_current BOOLEAN NOT NULL,                       -- Flag to easily find the current record

  -- Original Business Data Columns
  customerNumber INT NOT NULL,
  customerName STRING NOT NULL,
  contactLastName STRING NOT NULL,
  contactFirstName STRING NOT NULL,
  phone STRING NOT NULL,
  addressLine1 STRING NOT NULL,
  addressLine2 STRING,
  city STRING NOT NULL,
  state STRING,
  postalCode STRING,
  country STRING NOT NULL,
  salesRepEmployeeNumber INT,
  creditLimit DECIMAL(10,2),
  load_date TIMESTAMP NOT NULL  
);


                ### INITIAL LOAD customer###


In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import col, lit, current_date, when
from delta.tables import DeltaTable
source_df = spark.table("bronze_138080.customers")
# Keep only records where the phone number is not null
not_null_df = source_df.filter(col("phone").isNotNull())
deduped_df = not_null_df.dropDuplicates(["phone"])
records_to_insert = deduped_df.withColumn("is_current", lit(True)) \
                             .withColumn("eff_start_date", current_date()) \
                             .withColumn("load_date", current_date().cast("timestamp")) \
                             .withColumn("eff_end_date", lit(None).cast("date"))
final_columns_to_insert = [
  
  "eff_start_date",
  "eff_end_date", 
  "is_current",
  "customerNumber",
  "customerName",
  "contactLastName",
  "contactFirstName",
  "phone",
  "addressLine1",
  "addressLine2",
  "city",
  "state",
  "postalCode",
  "country",
  "salesRepEmployeeNumber",
  "creditLimit",
  "load_date"
]
final_insert_df = records_to_insert.select(final_columns_to_insert)
final_insert_df.write.format("delta").mode("append").saveAsTable("Silver_138080.customers")





    ###merge####

In [0]:
from pyspark.sql.functions import col, lit, current_date

table_name = "silver_138080.customers"
#updates_df = spark.table("bronze_138080.customers").where(col("phone").isNotNull())
source_df = spark.table("bronze_138080.customers").where(col("customerNumber").isin("128","227") & (col("phone").isNotNull()))
deduped_df = source_df.dropDuplicates(["phone"])
updates_df = deduped_df.withColumn(
    "phone",
    when(col("customerNumber") == 128, "999-9999")
    .when(col("customerNumber") == 227, "444-4444").otherwise(col("phone"))
)
# Use table name, not path, for Unity Catalog managed tables
from delta.tables import DeltaTable
delta_target = DeltaTable.forName(spark, table_name)
target_df = delta_target.toDF()

staged_updates_df = updates_df.alias("source").join(
    target_df.alias("target"),
    (col("source.customerNumber") == col("target.customerNumber")) &
    (col("target.is_current") == True) & 
    (col("source.phone") != col("target.phone") )
    ).select("source.*")

temp_path = "/mnt/data_analytics_training/customerstemp"
staged_updates_df.write.format("delta").mode("overwrite").save(temp_path)
# Read the data back from the temporary location. This new DataFrame is static.
materialized_staged_updates_df = spark.read.format("delta").load(temp_path)
print("Expiring old records...")

delta_target.alias("target").merge(
    source=staged_updates_df.alias("source"),
    condition="target.customerNumber = source.customerNumber AND target.is_current = True"
).whenMatchedUpdate(set={
    "is_current": "False",
    "eff_end_date": current_date()
}).execute()
changed_records_to_insert = materialized_staged_updates_df.withColumn("is_current", lit(True)) \
                                             .withColumn("eff_start_date", current_date()) \
                                             .withColumn("eff_end_date", lit(None).cast("date")) \
                                             .withColumn("load_date", current_date())


# Prepare the brand new customers (those in source but not in target at all)
new_customers_to_insert = updates_df.alias("source").join(
    target_df.alias("target"), "customerNumber", "left_anti"
).withColumn("is_current", lit(True)) \
 .withColumn("eff_start_date", current_date()) \
 .withColumn("eff_end_date", lit(None).cast("date")) \
 .withColumn("load_date", current_date().cast("timestamp"))

# Union and write the new rows
final_inserts_df = changed_records_to_insert.unionByName(new_customers_to_insert)
columns_to_select = [
    col for col in final_inserts_df.columns if col != "customer_sk"
]
df_no_sk = final_inserts_df.select(*columns_to_select)
display(df_no_sk)

print("Inserting new and updated records...")
df_no_sk.write.format("delta").mode("append").saveAsTable(table_name)


print("SCD Type 2 process completed.")



      ###Orderdetails###

In [0]:
%sql
CREATE TABLE if not EXISTS silver_138080.orderdetails (
  orderNumber INT NOT NULL,
  productCode VARCHAR(15) NOT NULL,
  quantityOrdered INT NOT NULL,
  priceEach DECIMAL(10,2) NOT NULL,
  orderLineNumber SMALLINT NOT NULL
);

In [0]:
%sql

insert into silver_138080.orderdetails
select * from bronze_138080.orderdetails

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(spark, "silver_138080.orderdetails")
updates_df= spark.table("bronze_138080.orderdetails")
print("Deduplicating source data based on (orderNumber, productCode)...")
window_spec = Window.partitionBy("orderNumber", "productCode").orderBy(col("orderLineNumber").desc())

print(f"Source record count before deduplication: {updates_df.count()}")
deduplicated_source_df = updates_df.withColumn("rank", row_number().over(window_spec)) \
                                     .filter(col("rank") == 1) \
                                     .drop("rank")
print(f"Source record count after deduplication: {deduplicated_source_df.count()}")
print("Deduplicated source data:")
#deduplicated_source_df.show()


# --- 3. MERGE Operation for SCD Type 1 Logic ---
print("\nPerforming SCD Type 1 merge...")
delta_target.alias("target").merge(
    source=deduplicated_source_df.alias("source"),
    condition="target.orderNumber = source.orderNumber AND target.productCode = source.productCode" # Join on the composite key
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("SCD Type 1 merge completed.")



In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Setup: Define Paths and Prepare Sample Data ---


# -- Initial State of the Silver Table (Target)

schema = ["orderNumber", "productCode", "quantityOrdered", "priceEach", "orderLineNumber"]


# -- New Batch of Data from Source (Bronze Layer)
# Note: Contains a duplicate for (orderNumber=10100, productCode="S18_1749")
updates_data = [
    (10100, "S18_1749", 35, 150.00, 3), # Updated quantity and price for an existing order line
    (10100, "S18_1749", 36, 151.00, 4), # Duplicate composite key with different data
    (10101, "Syyyyyy", 25, 108.00, 1)  # A brand new order line
]
updates_df = spark.createDataFrame(updates_data, schema)


# --- 2. Deduplicate Source Data by Composite Key ---

# We partition by the composite key and order by orderLineNumber to keep the latest entry.
print("Deduplicating source data based on (orderNumber, productCode)...")
window_spec = Window.partitionBy("orderNumber", "productCode").orderBy(col("orderLineNumber").desc())

deduplicated_source_df = updates_df.withColumn("rank", row_number().over(window_spec)) \
                                     .filter(col("rank") == 1) \
                                     .drop("rank")

print(f"Source record count after deduplication: {deduplicated_source_df.count()}")
print("Deduplicated source data:")
deduplicated_source_df.show()


# --- 3. MERGE Operation for SCD Type 1 Logic ---

# Load the target Delta table
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(spark, "silver_138080.orderdetails")

print("\nPerforming SCD Type 1 merge...")
delta_target.alias("target").merge(
    source=deduplicated_source_df.alias("source"),
    condition="target.orderNumber = source.orderNumber AND target.productCode = source.productCode" # Join on the composite key
).whenMatchedUpdateAll( # If the order line exists, OVERWRITE it with the new data
).whenNotMatchedInsertAll( # If the order line is new, INSERT it
).execute()

print("SCD Type 1 merge completed.")


# --- 4. Final Verification ---
print("\nFinal state of the silver.orderdetails table:")
display(
    spark.table("silver_138080.orderdetails").where((col("orderNumber") == 10100) & (col("productCode") == "S18_1749")).orderBy("orderNumber", "productCode")
)


      ####Check o duplicate Number
      

      ### Orders

In [0]:
%sql
CREATE TABLE if not EXISTS silver_138080.orders (
  orderNumber INT NOT NULL,
  orderDate DATE NOT NULL,
  requiredDate DATE NOT NULL,
  shippedDate DATE,
  status VARCHAR(15) NOT NULL,
  comments STRING,
  customerNumber INT NOT NULL
);

insert into silver_138080.orders 
Select orderNumber ,
  orderDate ,
  requiredDate ,
  shippedDate ,
  status ,
  comments ,
  customerNumber 
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER(PARTITION BY orderNumber ORDER BY orderDate DESC) as rn
    FROM
        bronze_138080.orders
) AS ranked_orders
WHERE
    rn = 1;


In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(spark, "silver_138080.orders")
updates_df= spark.table("bronze_138080.orders")
print("Deduplicating source data based on (orderNumber)...")
window_spec = Window.partitionBy("orderNumber").orderBy(col("orderDate").desc())

print(f"Source record count before deduplication: {updates_df.count()}")
deduplicated_source_df = updates_df.withColumn("rank", row_number().over(window_spec)) \
                                     .filter(col("rank") == 1) \
                                     .drop("rank")
print(f"Source record count after deduplication: {deduplicated_source_df.count()}")
print("Deduplicated source data:")
#deduplicated_source_df.show()


# --- 3. MERGE Operation for SCD Type 1 Logic ---
print("\nPerforming SCD Type 1 merge...")
delta_target.alias("target").merge(
    source=deduplicated_source_df.alias("source"),
    condition="target.orderNumber = source.orderNumber"# Join on the composite key
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("SCD Type 1 merge completed.")



      ###Payments####

In [0]:
%sql
CREATE TABLE if not exists silver_138080.payments (
  customerNumber INT NOT NULL,
  creditcardNumber VARCHAR(50) NOT NULL,
  paymentDate DATE NOT NULL,
  amount DECIMAL(10,2) NOT NULL
);
insert into silver_138080.payments 
Select
  customerNumber ,
  creditcardNumber ,
  paymentDate ,
  amount 
FROM bronze_138080.payments


In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(spark, "silver_138080.payments")
updates_df= spark.table("bronze_138080.payments")



# --- 3. MERGE Operation for SCD Type 1 Logic ---
print("\nPerforming SCD Type 1 merge...")
delta_target.alias("target").merge(
    source=updates_df.alias("source"),
    condition=("target.customerNumber = source.customerNumber AND "
        "target.creditcardNumber = source.creditcardNumber AND "
        "target.paymentDate = source.paymentDate AND "
        "target.amount = source.amount")
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("SCD Type 1 merge completed.")



      ###product#####

In [0]:
%sql
CREATE TABLE if not EXISTS Silver_138080.products (
  productCode VARCHAR(15) NOT NULL,
  productName VARCHAR(70) NOT NULL,
  productLine VARCHAR(50) NOT NULL,
  productScale VARCHAR(10) NOT NULL,
  productVendor VARCHAR(50) NOT NULL,
  productDescription STRING NOT NULL,
  quantityInStock SMALLINT NOT NULL,
  current_buyPrice DECIMAL(10,2) NOT NULL,
  previous_buyPrice DECIMAL(10,2),
  price_effective_date DATE,
  MSRP DECIMAL(10,2) NOT NULL
);

In [0]:
from pyspark.sql.functions import col, current_date, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- Setup: Define Paths and Prepare Sample Data ---

delta_target = DeltaTable.forName(spark, "silver_138080.products")
updates_df = spark.table("bronze_138080.products")

# --- Deduplicate Source Data by Product Code ---
print("Deduplicating source data based on productCode...")
window_spec = Window.partitionBy("productCode").orderBy(col("buyPrice").desc())
deduplicated_source_df = (
    updates_df
    .withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .drop("rank")
)

# --- MERGE Operation for SCD Type 3 Logic ---
print("\nPerforming SCD Type 3 merge...")
delta_target.alias("target").merge(
    source=deduplicated_source_df.alias("source"),
    condition="target.productCode = source.productCode"
).whenMatchedUpdate(
    condition="target.current_buyPrice <> source.buyPrice",
    set={
        "previous_buyPrice": "target.current_buyPrice",
        "current_buyPrice": "source.buyPrice",
        "price_effective_date": current_date(),
        "productName": "source.productName",
        "quantityInStock": "source.quantityInStock",
        "MSRP": "source.MSRP"
    }
).whenMatchedUpdate(
    condition="target.current_buyPrice = source.buyPrice",
    set={
        "productName": "source.productName",
        "quantityInStock": "source.quantityInStock",
        "MSRP": "source.MSRP"
    }
).whenNotMatchedInsert(
    values={
        "productCode": "source.productCode",
        "productName": "source.productName",
        "productLine": "source.productLine",
        "productScale": "source.productScale",
        "productVendor": "source.productVendor",
        "productDescription": "source.productDescription",
        "quantityInStock": "source.quantityInStock",
        "current_buyPrice": "source.buyPrice",
        "MSRP": "source.MSRP"
    }
).execute()

print("SCD Type 3 merge completed.")

# --- Final Verification ---
print("\nFinal state of the products table:")
display(
    spark.table("silver_138080.products").orderBy("productCode")
)

#Golden Schema


        ####table creation####

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold_138080;

CREATE TABLE gold_138080.KPI_Highest_Selling_product (
  productCode STRING NOT NULL,
  productName STRING NOT NULL,
  productVendor STRING NOT NULL,
  productDescription STRING NOT NULL,
  quantityInStock SMALLINT NOT NULL,
  buyPrice DECIMAL(10,2) NOT NULL,
  MSRP DECIMAL(10,2) NOT NULL,
  month STRING NOT NULL
);

CREATE TABLE gold_138080.KPI_Highest_Spending_Customer (
  customerNumber INT NOT NULL,
  customerName STRING NOT NULL,
  contactLastName STRING NOT NULL,
  contactFirstName STRING NOT NULL,
  phone STRING NOT NULL,
  addressLine1 STRING NOT NULL,
  addressLine2 STRING,
  month STRING NOT NULL
);

CREATE TABLE gold_138080.KPI_Highest_Order_City (
  city STRING NOT NULL,
  state STRING,
  amount DECIMAL(10,2) NOT NULL,
  month STRING NOT NULL
);

        ####KPI_Highest_Selling_product####

In [0]:
%sql
WITH cte AS (
  select a.productCode,Sum(a.quantityOrdered) as qorder,MonthName(b.orderDate) as MonthName  from Silver_138080.orderdetails a
inner join Silver_138080.orders b on a.orderNumber=b.orderNumber
 group by MonthName(b.orderDate),a.productCode
)
SELECT 
  p.productCode,
  p.productName,
  p.productVendor,
  p.productDescription,
  p.quantityInStock,
  p.current_buyPrice as buyPrice ,
  p.MSRP,
  MonthName
FROM cte
INNER JOIN silver_138080.products p
  ON cte.productCode = p.productCode
ORDER BY p.productCode

In [0]:
%sql
;WITH cte AS (
 select a.productCode,Sum(a.quantityOrdered) as qorder,MonthName(b.orderDate) as MonthName  from Silver_138080.orderdetails a
inner join Silver_138080.orders b on a.orderNumber=b.orderNumber 
 group by MonthName(b.orderDate),a.productCode
),
cte2 as( SELECT productCode,Qorder,rank() over(partition by monthname order by qorder desc) as rn,MonthName from cte  )
insert into gold_138080.KPI_Highest_Selling_product
SELECT 
  p.productCode,
  p.productName,
  p.productVendor,
  p.productDescription,
  p.quantityInStock,
  p.current_buyPrice as buyPrice ,
  p.MSRP,
  MonthName
FROM cte2
INNER JOIN silver_138080.products p
  ON cte2.productCode = p.productCode and rn=1
ORDER BY MonthName

 

          ####KPI_Highest_Spending_Customer####

In [0]:
%sql
;with cte(
Select  customerNumber,Sum(amount) amount,monthname(paymentDate) as monthname from silver_138080.payments 
group by customerNumber,monthname(paymentDate))
,cte2(Select customerNumber,rank () over(partition by monthname order by amount desc) as rn,amount,monthname from cte)
insert into gold_138080.KPI_Highest_Spending_Customer
Select a.customerNumber,a.customerName,a.contactLastName,a.contactFirstName,a.phone,a.addressLine1,a.addressLine2,monthname from
silver_138080.customers a inner join  cte2 b on a.customerNumber=b.customerNumber where rn=1 and a.is_current=true
    


In [0]:
%sql
select a.productCode,Sum(a.quantityOrdered) as qorder,MonthName(b.orderDate) as MonthName  from Silver_138080.orderdetails a
inner join Silver_138080.orders b on a.orderNumber=b.orderNumber where a.productCode="S18_4933"
 group by MonthName(b.orderDate),a.productCode

      ####KPI_Highest_Order_City####

In [0]:
%sql
WITH cte as (
Select b.customerNumber,sum(a.quantityOrdered*a.priceEach)amount,monthname(orderdate) monthname from silver_138080.orderdetails a
inner join silver_138080.orders b on a.orderNumber=b.orderNumber 
group by b.customerNumber,monthname(orderdate) )
,cte2(Select *, rank() over (partition by monthname order by amount desc) rnk from cte )
,cte3(Select * from cte2 where rnk=1)
insert into gold_138080.KPI_Highest_Order_City
Select b.city,b.state,amount,monthname from cte3 a inner join silver_138080.customers b on a.customernumber=b.customerNumber where b.is_current=true